# BSM Paper Replication — Notebook 05: D-ResNet-50 (paper-faithful, real run)

SMOKE_TEST=False. 60 epochs, full-batch grad-accum (micro 32, one optim step/epoch), Adam 1e-3. Best-val checkpoint. Evaluate full test split. Stop if train_acc < 0.30 at epoch 30.

**Param count note:** `dresnet50(head='standard')` = 25,557,032 (matches torchvision ResNet-50; used only for the FLOP/param verification in nb01). The paper head (`paper_fc1` 2048->1000 + ReLU + `paper_fc2` 1000->6) = **25,563,038** (+6,006) — this is the model actually trained here.

In [1]:
# setup
import os, sys
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)
import torch, random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
SMOKE_TEST = False
print("gpu:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "SMOKE_TEST=", SMOKE_TEST)


gpu: True Tesla T4 SMOKE_TEST= False


## Real run: D-ResNet-50, 60 epochs, full-batch grad-accum (micro 32, one optim step/epoch), Adam 1e-3, best-val checkpoint

In [2]:
# ===== REAL TRAINING: D-ResNet-50 (60 epochs, paper-faithful) =====
import os, sys, time, json, pathlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = pathlib.Path(PROJECT) / "versions"
CKPT = pathlib.Path(PROJECT) / "checkpoints" / "dresnet50"
RES  = pathlib.Path(PROJECT) / "results"
CKPT.mkdir(parents=True, exist_ok=True)
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)

from models import dresnet50
import common

# ---- data (split arrays from nb04) ----
def load_xy(prefix):
    X = np.load(VER / f"X_{prefix}.npy")
    y = np.load(VER / f"y_{prefix}.npy")
    Xt = torch.from_numpy(X.transpose(0, 3, 1, 2)).contiguous()  # NHWC -> NCHW
    yt = torch.from_numpy(y.astype(np.int64))
    return Xt, yt

Xtr, ytr = load_xy("train")
Xva, yva = load_xy("val")
print("train:", Xtr.shape, "val:", Xva.shape, "| dtype", Xtr.dtype, "range", float(Xtr.min()), float(Xtr.max()))

MICRO = 32                      # micro-batch for gradient accumulation
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=MICRO, shuffle=False)
val_loader   = DataLoader(TensorDataset(Xva, yva), batch_size=MICRO, shuffle=False)

# ---- model ----
torch.manual_seed(42)
model = dresnet50().cuda()
n_params = sum(p.numel() for p in model.parameters())
print("dresnet50 params:", f"{n_params:,}")

EPOCHS = 60
t0 = time.time()
hist = common.train_model(
    model, train_loader, val_loader, torch.device("cuda"),
    epochs=EPOCHS, full_batch=True,          # grad accum over full set: 1 optim step/epoch
    ckpt_dir=CKPT, history_path=RES / "history_dresnet50.json",
    tag="dresnet50", lr=1e-3, resume=True,
)
print(f"TRAINING DONE in {(time.time()-t0)/60:.1f} min")
print("best epoch:", hist.get("best_epoch"), "best val:", hist.get("best_val_acc"))

# ---- guard: stop-reason check (spec: train acc < 0.30 at epoch 30) ----
tr_acc = hist.get("train_acc") or []
if len(tr_acc) >= 30 and tr_acc[29] < 0.30:
    print("WARNING: train acc at epoch 30 =", tr_acc[29], "< 0.30 (spec stop-check)")


train: torch.Size([5651, 3, 224, 224]) val: torch.Size([1211, 3, 224, 224]) | dtype torch.float32 range 0.0 1.0
dresnet50 params: 25,563,038
dresnet50 epoch 01/60  train_loss 1.8362 acc 0.1794  val_loss 482.6707 acc 0.1685  (26.4s)
dresnet50 epoch 02/60  train_loss 5.5416 acc 0.1690  val_loss nan acc 0.1618  (23.0s)
dresnet50 epoch 03/60  train_loss 3.4532 acc 0.1766  val_loss 117.1906 acc 0.1635  (23.4s)
dresnet50 epoch 04/60  train_loss 3.1374 acc 0.1649  val_loss 13.4209 acc 0.1825  (23.9s)
dresnet50 epoch 05/60  train_loss 2.7064 acc 0.2350  val_loss 3.8096 acc 0.2056  (24.5s)
dresnet50 epoch 06/60  train_loss 2.2738 acc 0.1747  val_loss 2.5054 acc 0.1800  (24.4s)
dresnet50 epoch 07/60  train_loss 2.0950 acc 0.1798  val_loss 2.4406 acc 0.1800  (24.1s)
dresnet50 epoch 08/60  train_loss 2.1198 acc 0.1798  val_loss 2.2094 acc 0.1800  (24.1s)
dresnet50 epoch 09/60  train_loss 2.0667 acc 0.1798  val_loss 2.0393 acc 0.1800  (24.2s)
dresnet50 epoch 10/60  train_loss 1.9957 acc 0.1798  val

### Re-engineering round (DEVIATIONS row 14)

If the train-vs-validation gap at the best-validation epoch exceeds 0.15, ONE
extra round is trained from a fresh model with lr 5e-4 and weight decay 1e-4, and
the round with the higher validation accuracy is the one evaluated on the test
split. The threshold and the round-1 hyper-parameters are our documented choice
(the paper does not define them).

In [ ]:
# --- RE-ENGINEERING ROUND (spec: re-engineering if train - val gap > 0.15) ---
# Paper's Fig. 5 "model re-engineering" loop. The 0.15 threshold and the
# round-1 hyper-parameters (lr 5e-4, weight decay 1e-4) are OUR documented
# choice -- the paper does not define them (DEVIATIONS row 14).
import json as _json
import numpy as _np
import torch as _torch

RE_ENGINEER_GAP = 0.15
RE_ENGINEER_LR, RE_ENGINEER_WD = 5e-4, 1e-4
RE_ENGINEER_EPOCHS = 60

def _round_stats(tag, history_file):
    h = _json.load(open(history_file))
    b = int(_np.argmax(h["val_acc"])) + 1
    return {"tag": tag, "best_epoch": b,
            "best_val": float(h["val_acc"][b - 1]),
            "train_at_best": float(h["train_acc"][b - 1]),
            "train_seconds": float(_np.sum(h.get("epoch_time_s", []))),
            "gap": float(h["train_acc"][b - 1] - h["val_acc"][b - 1])}

ROUNDS = [_round_stats("dresnet50", RES / "history_dresnet50.json")]
print("round 0:", ROUNDS[0])

if ROUNDS[0]["gap"] > RE_ENGINEER_GAP:
    print(f"gap {ROUNDS[0]['gap']:+.4f} > {RE_ENGINEER_GAP} -> RE-ENGINEERING ROUND 1 "
          f"(fresh model, lr {RE_ENGINEER_LR}, wd {RE_ENGINEER_WD}, {RE_ENGINEER_EPOCHS} epochs)")
    _torch.manual_seed(SEED)
    model_re1 = dresnet50().cuda()
    common.train_model(
        model_re1, train_loader, val_loader, _torch.device("cuda"),
        epochs=RE_ENGINEER_EPOCHS, full_batch=True,
        ckpt_dir=CKPT, history_path=RES / "history_dresnet50_re1.json",
        tag="dresnet50_re1", lr=RE_ENGINEER_LR, wd=RE_ENGINEER_WD, resume=True,
    )
    ROUNDS.append(_round_stats("dresnet50_re1", RES / "history_dresnet50_re1.json"))
    print("round 1:", ROUNDS[1])
else:
    print(f"gap {ROUNDS[0]['gap']:+.4f} <= {RE_ENGINEER_GAP} -> no re-engineering round needed")

SELECTED_ROUND = max(ROUNDS, key=lambda r: r["best_val"])
print("SELECTED ROUND (higher validation accuracy):", SELECTED_ROUND)
with open(RES / "selected_round_dresnet50.json", "w") as f:
    _json.dump({"rounds": ROUNDS, "selected": SELECTED_ROUND}, f, indent=2)


In [3]:
# ===== EVAL D-ResNet-50 (best-val ckpt) on TEST SPLIT =====
import os, sys, json, pathlib, time
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = pathlib.Path(PROJECT) / "versions"
CKPT = pathlib.Path(PROJECT) / "checkpoints" / "dresnet50"
RES  = pathlib.Path(PROJECT) / "results"
RES.mkdir(parents=True, exist_ok=True)
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)
from models import dresnet50
import common

CLASSES = ["Ilmenite","Rutile","Zircon","Garnet","Monazite","Sillimanite"]

def load_xy(prefix):
    X = np.load(VER / f"X_{prefix}.npy")
    y = np.load(VER / f"y_{prefix}.npy")
    return torch.from_numpy(X.transpose(0,3,1,2)).contiguous(), torch.from_numpy(y.astype(np.int64))

Xte, yte = load_xy("test")
print("test:", Xte.shape)
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=32, shuffle=False)

device = torch.device("cuda")
model = dresnet50().to(device)
print("evaluating selected round:", SELECTED_ROUND)
ck = torch.load(CKPT / f"{SELECTED_ROUND['tag']}_best.pt", map_location=device, weights_only=True)
model.load_state_dict(ck)
import json as _json
_ck = {"epoch": SELECTED_ROUND["best_epoch"]}
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(device))
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_true.append(yb.numpy())
y_pred = np.concatenate(all_preds); y_true = np.concatenate(all_true)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=range(6))
# specificity (macro): TN/(TN+FP) per class
spec_list = []
for i in range(6):
    tn = cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]
    fp = cm[:,i].sum() - cm[i,i]
    spec_list.append(tn/(tn+fp) if (tn+fp)>0 else 0.0)
spec = float(np.mean(spec_list))

print(f"\n=== D-ResNet-50 TEST SPLIT (best epoch {ck.get('epoch','?')}) ===")
print(f"accuracy {acc:.4f} | macro precision {prec:.4f} | macro recall {rec:.4f} | macro F1 {f1:.4f} | macro specificity {spec:.4f}")
print("\nconfusion matrix (rows=true):")
print(cm)
print("\nper-class:")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0))

results = {
  "model": "dresnet50", "notebook": "05_dresnet50",
  "best_epoch": int(_ck.get("epoch", -1)),
  "selected_round": SELECTED_ROUND,
  "re_engineering": ROUNDS,
  "train_minutes": round(sum(r.get("train_seconds", 0.0) for r in ROUNDS) / 60, 2),
  "test_split": {"accuracy": float(acc), "macro_precision": float(prec), "macro_recall": float(rec),
                 "macro_f1": float(f1), "macro_specificity": spec,
                 "confusion_matrix": cm.tolist(), "n_test": int(len(y_true))},
  "paper_reported_table9": {"accuracy": 0.89, "precision": 0.90, "recall": 0.89, "note": "PAPER numbers (Table 9), for comparison only"},
}
with open(RES / "metrics_dresnet50.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nsaved", RES / "metrics_dresnet50.json")
print("EVAL DONE")


test: torch.Size([1212, 3, 224, 224])

=== D-ResNet-50 TEST SPLIT (best epoch ?) ===
accuracy 0.7228 | macro precision 0.7411 | macro recall 0.7244 | macro F1 0.7206 | macro specificity 0.9446

confusion matrix (rows=true):
[[194   1   5   3   0   2]
 [ 86 117   1   0   4  10]
 [ 11   7 130  17  17  13]
 [  9   3  43 129   5   9]
 [  0   4   4   4 159  27]
 [ 17  15   5   1  13 147]]

per-class:
              precision    recall  f1-score   support

    Ilmenite     0.6120    0.9463    0.7433       205
      Rutile     0.7959    0.5367    0.6411       218
      Zircon     0.6915    0.6667    0.6789       195
      Garnet     0.8377    0.6515    0.7330       198
    Monazite     0.8030    0.8030    0.8030       198
 Sillimanite     0.7067    0.7424    0.7241       198

    accuracy                         0.7228      1212
   macro avg     0.7411    0.7244    0.7206      1212
weighted avg     0.7414    0.7228    0.7195      1212


saved /content/drive/MyDrive/bsm_paper_replication/result

In [4]:
# persist notebook copy
import glob, os, shutil
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
os.makedirs(os.path.join(PROJECT,"notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    shutil.copy(newest, os.path.join(PROJECT,"notebooks","05_dresnet50.ipynb"))
    print("copied ->", os.path.join(PROJECT,"notebooks","05_dresnet50.ipynb"))


copied -> /content/drive/MyDrive/bsm_paper_replication/notebooks/05_dresnet50.ipynb
